<div style="background-color: lightblue; color: black; border: 2px solid black; 
padding: 15px 25px; border-radius: 8px; margin: 10px 0px;">

  <h2 style="color: black; margin: 0px 0px 10px 0px;">
   Testing for Unphysical Precipitation Values in UKCP Local
  </h2>

  <hr style="border: 1px solid black; margin: 10px 0px;">

  <p>In the set of extreme events extracted from UKCP Local (bias-corrected, 
  5km data), some events have maximum precipitation values which are 
  unphysical — i.e., higher than is physically plausible.</p>

  <p><strong>This script tests:</strong></p>
  <ul>
    <li>The number of events per catchment, per ensemble member with an unphysical value</li>
    <li>The number of pixels affected per event</li>
  </ul>

  <div style="background-color: white; border-left: 4px solid orange; 
  padding: 8px 15px; border-radius: 0px 5px 5px 0px; margin-top: 12px;">
    <strong>NB:</strong> The threshold for an "unphysical" value is 
    currently set at <strong>130mm/hr</strong> — based on the max value in the non-bc data
  </div>

</div>

In [1]:
import numpy as np
import pandas as pd
import time
import os
import sys
import glob
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from collections import Counter
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import importlib
import ast
import iris

import Functions_HighValsCheck
importlib.reload(Functions_HighValsCheck)

from Functions_HighValsCheck import (find_problematic_catchment_em_combos_complex, check_extremes_across_combos,
                                    plot_surrounding_ts_new, plot_n_peaks, compare_bc_nonbc_timeseries)

# ── Config ────────────────────────────────────────────────────────────────────
from config import RAINFALL_CSV_DIR, CATCHMENT_LOOKUP_DICT, OUT_DIR, CATCHMENTS, ENSEMBLE_MEMBERS 

# ── Define which catchments to run ────────────────────────────────────────────────────────────────────
all_catchments = set(CATCHMENT_LOOKUP_DICT.keys())

<div style="background-color: lightblue; color: black;  border: 2px solid black; padding: 15px 20px; border-radius: 5px; margin: 10px 0px;"><h3 style="color: black; margin: 0px;"> Find catchment, ensemble member combinations with values >150mm/hr </h3>

  <p>For each catchment–ensemble member combination:</p>
  <ul>
    <li>Find the number of events exceeding the <strong>130mm/hr</strong> threshold</li>
    <li>Calculate this as a proportion of the total number of events</li>
  </ul>
    
<div style="background-color: white; border-left: 4px solid orange; 
  padding: 8px 15px; border-radius: 0px 5px 5px 0px; margin-top: 12px;">
This shows that for affected catchments the number of events affected are not insignificant: of the 66 instances of a catchment/ensemble member combination which appear in results_df ~50% have more than 10% of the events with unphysical values, and some have up to almost 50%.    
  </div>    

  <p>Also finds, for the highest peak value in each catchment:</p>
  <ul>
    <li>The number of timesteps which are unphysical (in any catchment cell) </li>
    <li>The number of cells which experience any unphysical values within 10 timesteps of the highest unphysical value </li>
    <li>The total number of cells in any cell, at any timestep (within 10 timesteps of the highest unphysical value) which are unphysical </li>     
      <li> The maximum number of cells experiencing an unphysical value concurrently </li>
      <li> The maximum number of timesteps for one cell which experience an unphysical value
  </ul>
<div style="background-color: white; border-left: 4px solid orange; 
padding: 8px 15px; border-radius: 0px 5px 5px 0px; margin-top: 12px;">
This shows that the unphysical values do not always occur in isolation. There are frequently either (i) other grid cells nearby affected, or (ii) other timesteps affected before or after.
   
  </div>   
</div>

In [2]:
if os.path.isfile("../../../Outputs/ProblematicCatchments_130Threshold.csv"):
        results_df = pd.read_csv("../../../Outputs/ProblematicCatchments_130Threshold.csv")
        problematic_catchment_nums = results_df['Catchment No.']
else:
    results_df, problematic_catchment_names, problematic_catchment_nums = find_problematic_catchment_em_combos_complex(all_catchments, 130, CATCHMENTS, CATCHMENT_LOOKUP_DICT, ENSEMBLE_MEMBERS, RAINFALL_CSV_DIR)
    results_df.to_csv("../../../Outputs/ProblematicCatchments_130Threshold.csv")
# results_df.sort_values(by='% Problematic', ascending = False)[:5]

In [3]:
# Get list of catchment_num, ensemble_member combos
problematic_combos  =[]
for num, row in results_df.iterrows():
    problematic_combos.append((row['Catchment No.'], row['Ensemble Member']))

In [4]:
# Check number of problematic events
lengths = 0
for row in results_df["Problematic Event Nums"]:
    lengths+=len(row)

<div style="background-color: lightblue; color: black;  border: 2px solid black; padding: 15px 20px; border-radius: 5px; margin: 10px 0px;"><h3 style="color: black; margin: 0px;"> Check of corresponding daily totals </h3>


This cell re-examines rainfall events previously flagged as **problematic** (e.g. implausibly
high hourly totals) in `results_df`, for the first 5 catchment/ensemble-member combinations.

For each flagged combination, it:

1. **Loads supporting data**, using caches to avoid redundant disk reads:
   - The **daily** rainfall cube for the relevant ensemble member.
   - A **catchment mask** (`FULL_MASK_2D`) identifying which grid cells fall inside the
     catchment boundary.
   - The **events table** for that catchment/ensemble member, with a `hydro_year` column
     added (December-start events are shifted into the following calendar year).

2. **For each problematic event**, extracts the corresponding hourly rainfall time series
   from both the **bias-corrected (BC)** and **non-bias-corrected (non-BC)** cubes, masked
   to the catchment. For each version it computes:
   - The peak rainfall value across the catchment and event duration (`max_bc` / `max_nonbc`).
   - The index of the peak hour within the event (`peak_idx_bc` / `peak_idx_nonbc`).
   - The full per-hour catchment-max time series (`ts_bc` / `ts_nonbc`).
   - The calendar days the event spans (`event_days`).

   Comparing `max_bc` vs `max_nonbc` helps identify whether bias correction is introducing
   or amplifying implausible spikes.

3. **Cross-checks against daily data**: for each day the event spans, extracts that day's
   value from the independent daily rainfall cube and prints its maximum, as a sanity check
   against the hourly-derived peaks.


In [8]:
results_df_orig = results_df.copy()

In [9]:
from config import *
from functions import get_rainfall_event_details, mask_cube_with_catchment_full_grid, subset_cube_to_bbox
from functions_stage2 import (get_rainfall_cube_subsection, get_exceedance_summary, find_max_precip_location_new, get_data_at_peak_cell,
                                find_temporal_profile_new)
from Functions_HighValsCheck import get_rainfall_cube_subsection_notbc

In [23]:
results_df = results_df_orig[1:2]
results_df.reset_index(inplace=True, drop=True)
ens_num = results_df['Ensemble Member'][0]

In [29]:
rows = []

# ── CACHES ─────────────────────────────────────────────
cube_cache   = {}   # (year, ens_num, bc) → cube
mask_cache   = {}   # catchment_num → FULL_MASK_2D
events_cache = {}   # (catchment_name, ens_num) → dataframe

def get_cube(year, ens_num, cube_dir, bc):
    key = (year, ens_num, bc)

    if key not in cube_cache:
        loader = get_rainfall_cube_subsection if bc else get_rainfall_cube_subsection_notbc
        cube_cache[key] = loader(year, ens_num, cube_dir, None, None)

    return cube_cache[key]

for _, combo_row in results_df.iterrows():

    catchment_num  = combo_row['Catchment No.']
    ens_num        = combo_row['Ensemble Member']
    ens_num = f"{int(ens_num):02d}"
    print(f" Catchment number: {catchment_num}, EM: {ens_num}")

    daily = iris.load(f"/scratch/hydro5/users/ld14116/SDM_bias_correction/{str(ens_num).zfill(2)}/1991_2080.nc")[0]
    
    rainfall_cube_dir_bc = f"/scratch/hydro5/users/ld14116/SDM_bias_correction/Hourly/{ens_num}/"
    rainfall_cube_dir_nonbc = f"/scratch/hydro4/shared_data/climate_projections/UKCP18/UKCP_local/Precipitation/Hourly/5km/Ens_{ens_num}/"    

    event_nums_to_check = ast.literal_eval(combo_row['Problematic Event Nums'])
    print(event_nums_to_check)
    catchment_name = CATCHMENT_LOOKUP_DICT[str(catchment_num)]

    # ── MASK CACHE ─────────────────────────────────────
    if catchment_num not in mask_cache:
        boundary_gdf   = CATCHMENTS[CATCHMENTS['HA_NUM'] == str(catchment_num)]
        CATCHMENT_POLY = boundary_gdf.geometry.iloc[0]

        full_rain_cube = get_rainfall_cube_subsection( 2015, '01',
            "/scratch/hydro5/users/ld14116/SDM_bias_correction/Hourly/01/", 1, 2)

        mask_cache[catchment_num] = mask_cube_with_catchment_full_grid(
            full_rain_cube[0], CATCHMENT_POLY, method='center_point')

    FULL_MASK_2D = mask_cache[catchment_num]

    # ── EVENTS CACHE ───────────────────────────────────
    events_key = (catchment_name, ens_num)

    if events_key not in events_cache:
        df = pd.read_csv(
            RAINFALL_CSV_DIR + f"{catchment_name}_{ens_num}_full_events_with_event_nums.csv")
        df['event_num']  = range(1, len(df) + 1)
        df['hydro_year'] = df['start_year'].where(
            df['start_month'] != 12, df['start_year'] + 1)

        df = df.set_index('event_num')
        events_cache[events_key] = df

    rainfall_events = events_cache[events_key]

    for event_num in event_nums_to_check:
        print(" --------- ")
        print(f"Event num: {event_num}")

        event_row = rainfall_events.loc[event_num]
        year      = int(event_row['hydro_year'])

        event_details = get_rainfall_event_details(
            rainfall_events.reset_index(), event_num)
        
        start_idx = event_details['start_idx']
        stop_idx  = event_details['stop_idx']

        def get_event_series(cube_dir, bc):
            cube_full = get_cube(year, ens_num, cube_dir, bc)
            cube = cube_full[start_idx:stop_idx]
            #print(cube.coord('time').units.calendar)
            #print(daily.coord('time').units.calendar)
            data = np.where(FULL_MASK_2D, cube.data, np.nan)

            ts_max   = np.nanmax(data.reshape(data.shape[0], -1), axis=1)
            peak_idx = int(np.argmax(ts_max))
            
            dates = pd.to_datetime(cube.coord('yyyymmddhh').points.astype(str), format="%Y%m%d%H")
            event_days = dates.normalize().unique()
            
            return float(np.max(ts_max)), peak_idx, ts_max, event_days

        max_bc, peak_idx_bc, ts_bc, event_days = get_event_series(rainfall_cube_dir_bc, bc=True)
        max_nonbc, peak_idx_nonbc, ts_nonbc, event_days = get_event_series(rainfall_cube_dir_nonbc, bc=False)
        print(f"Max BC: {max_bc}")
        print(f"Max not BC: {max_nonbc}")
        
        for day in event_days:
            day_str = day.strftime("%Y%m%d")
        
            #yyyymmdd_str = f"{event_details['yr']}{event_details['month']}{str(event_details['day']).zfill(2)}"
            #print(yyyymmdd_str)
            cube_1day = daily.extract(iris.Constraint(yyyymmdd=lambda cell: cell == day_str))
            print(f"Day: {day_str}, Daily max: {np.nanmax(cube_1day.data)}")
            


 Catchment number: 104, EM: 05
[12, 75, 133, 135, 138, 139, 141, 146, 155, 156, 162, 163, 165, 166, 172, 176, 182, 194]
 --------- 
Event num: 12
Max BC: 142.11865234375
Max not BC: 7.1478776931762695
Day: 20171206, Daily max: 3.7262508869171143
Day: 20171207, Daily max: 7.32656192779541
 --------- 
Event num: 75
Max BC: 138.7386016845703
Max not BC: 4.12021017074585
Day: 20261223, Daily max: 6.959446430206299
Day: 20261224, Daily max: 9.246971130371094
 --------- 
Event num: 133
Max BC: 153.12744140625
Max not BC: 18.680627822875977
Day: 20351218, Daily max: 4.412318229675293
Day: 20351219, Daily max: 2.423935890197754
Day: 20351220, Daily max: 2.6829113960266113
Day: 20351221, Daily max: 9.801124572753906
Day: 20351222, Daily max: 1.8989073038101196
 --------- 
Event num: 135
Max BC: 199.65472412109375
Max not BC: 5.591643333435059
Day: 20361208, Daily max: 8.671713829040527
Day: 20361209, Daily max: 10.660331726074219
 --------- 
Event num: 138
Max BC: 158.11451721191406
Max not BC:

KeyboardInterrupt: 